In [ ]:
"""
Label patch — fixes FTP Patator / SSH Patator mapping
and cleans up None tactic on benign rows.
to be run after the initial processing
"""
import pandas as pd
from pathlib import Path

PATH = Path("../data/processed/cicids_processed.csv")

print("Loading processed file...")
df = pd.read_csv(PATH, low_memory=False, index_col="alert_id")
print(f"  Rows: {len(df):,}")

# ── Fix 1: FTP Patator / SSH Patator ────────────────────────────────────────
mask_ftp = df["Label"] == "FTP Patator"
mask_ssh = df["Label"] == "SSH Patator"

df.loc[mask_ftp, "attck_technique_id"]   = "T1110.001"
df.loc[mask_ftp, "attck_technique_name"] = "Password Guessing"
df.loc[mask_ftp, "attck_tactic"]         = "Credential Access"

df.loc[mask_ssh, "attck_technique_id"]   = "T1110.001"
df.loc[mask_ssh, "attck_technique_name"] = "Password Guessing"
df.loc[mask_ssh, "attck_tactic"]         = "Credential Access"

print(f"  Fixed FTP Patator rows : {mask_ftp.sum():,}")
print(f"  Fixed SSH Patator rows : {mask_ssh.sum():,}")

# ── Fix 2: Benign tactic label ───────────────────────────────────────────────
mask_benign = df["Label"] == "BENIGN"
df.loc[mask_benign, "attck_tactic"] = "Benign"
print(f"  Fixed Benign tactic    : {mask_benign.sum():,}")

# ── Fix 3: Also update alert_text for patator rows ──────────────────────────
def fix_patator_text(row):
    if row["Label"] in ["FTP Patator", "SSH Patator"]:
        return row["alert_text"].replace(
            "Traffic classified as FTP Patator.",
            "Traffic classified as FTP-Patator. ATT&CK technique T1110.001 (Credential Access)."
        ).replace(
            "Traffic classified as SSH Patator.",
            "Traffic classified as SSH-Patator. ATT&CK technique T1110.001 (Credential Access)."
        )
    return row["alert_text"]

df["alert_text"] = df.apply(fix_patator_text, axis=1)

# ── Save ─────────────────────────────────────────────────────────────────────
df.to_csv(PATH)
print(f"\nPatched file saved to: {PATH}")

# ── Verify ───────────────────────────────────────────────────────────────────
print(f"\nFinal tactic distribution:")
print(df["attck_tactic"].value_counts().to_string())
print(f"\nFinal technique distribution:")
print(df["attck_technique_id"].value_counts().to_string())
unmapped = df[df["attck_technique_id"].isin(["UNMAPPED", "Unknown"])]
if len(unmapped) > 0:
    print(f"\nWARNING: {len(unmapped)} still unmapped:")
    print(unmapped["Label"].value_counts())
else:
    print(f"\nAll labels successfully mapped.")

Loading processed file...
  Rows: 435,290
  Fixed FTP Patator rows : 7,938
  Fixed SSH Patator rows : 5,897
  Fixed Benign tactic    : 130,316

Patched file saved to: ../data/processed/cicids_processed.csv

Final tactic distribution:
attck_tactic
Discovery              158930
Benign                 130316
Impact                 128027
Credential Access       15342
Command And Control      2002
Execution                 652
Initial Access             21

Final technique distribution:
attck_technique_id
T1046        158930
BENIGN       130316
T1498.001    128027
T1110.001     15342
T1071.001      1966
T1059.007       652
T1105            36
T1190            21

All labels successfully mapped.
